# MLB Trade Value Engine
### A data-driven framework for pricing players at the trade deadline

*Zach Thomas*

---

Major league trades involve millions of dollars, years of organizational talent, and career-defining decisions — made in days, sometimes hours. This project builds a systematic framework for one core question: **given a player's profile, what return should a team expect?**

The model works in two parts:
1. **Surplus Value** — how much production does the player provide above their salary cost, discounted for time and controllability?
2. **Historical Comps** — what did teams actually receive when they traded similar players?

The database covers **370 verified MLB trades (2015–2026)** with full WAR history, contract context, and return grades.

## The Framework

Baseball's accepted currency for player value is **WAR** (Wins Above Replacement) — a single number capturing a player's total contribution relative to a freely available replacement. One WAR is worth roughly **$7M on the open market** (calibrated from 2022–2025 free agent contracts).

**Surplus value** is the gap between what a player produces and what they cost:

> *Surplus = (WAR × $/WAR) − Salary*

A pre-arb player earning $750K who produces 5 WAR generates ~$34M in surplus value. A free agent earning $20M who produces 2 WAR generates almost none.

Three factors adjust the raw number:
- **Discount rate (5%/yr)** — future production is worth less than current production
- **Controllability (0.875×)** — team control is more valuable than a comparable free agent because the player cannot opt out
- **Contract risk** — salary obligations past peak years create a negative surplus drag

**WAR projection** uses Baseball Reference bWAR (annualized from YTD pace at the current team games-played count) as the primary source. FanGraphs leaderboard APIs have been inaccessible via automation since mid-2026; the model falls back to local FanGraphs projection files (THE BAT X / ZiPS) when live bWAR data is unavailable. Pitchers are annualized against team games played, not pitcher appearances.

**Calibration.** Tier thresholds and development discounts were validated against all 370 trades in the database (back-test: r = 0.538, MAE = 1.60 tiers). Pre-arb and early-arb players carry an additional development discount (0.70–0.80×) and a WAR-floor ceiling derived from the empirical p90 of actual return tiers — because the surplus formula is structurally optimistic for unproven players. For those profiles, historical comps carry more weight than the surplus tier.

The result is a **Net Trade Tier (1–10)** combining talent quality and contract favorability. This tier anchors the historical comp search.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

TRADES_CSV = Path.home() / 'projects/trade-value-engine/trades.csv'
df = pd.read_csv(TRADES_CSV, encoding='utf-8-sig')
df = df.rename(columns={df.columns[0]: 'trade_id'})

# Only complete rows (have return_tier, age, salary)
complete = df[
    df['return_tier'].notna() &
    df['age_at_trade'].notna() &
    df['salary_m'].notna()
].copy()
complete['return_tier'] = complete['return_tier'].astype(int)
complete['wWAR'] = pd.to_numeric(complete['wWAR'], errors='coerce')

print(f"Trade database: {len(complete)} verified trades, "
      f"{complete['season'].nunique()} seasons "
      f"({int(complete['season'].min())}–{int(complete['season'].max())})")

Trade database: 368 verified trades, 11 seasons (2015–2026)


## The Trade Database

370 verified trades across 2015–2026, spanning every major position, contract status, and team situation. Each entry includes:
- WAR history (3 prior seasons, Marcel-weighted into **wWAR**)
- Contract status, salary, and years of control
- Full return package with prospect grades and a 1–10 return tier

Return tiers are graded based on **prospect rankings at the time of trade**, not hindsight — tier 8 in 2022 means the prospects were nationally ranked then, not what they became.

In [2]:
tier_labels = {
    10: 'Franchise-altering', 9: 'Elite package', 8: 'Very high',
    7: 'High', 6: 'Mid-high', 5: 'Mid', 4: 'Mid-low',
    3: 'Depth', 2: 'Minimal', 1: 'Salary dump',
}
tier_dist = complete['return_tier'].value_counts().sort_index()
summary = pd.DataFrame({
    'Return Tier': tier_dist.index,
    'Description': [tier_labels.get(t, '') for t in tier_dist.index],
    'Trade Count': tier_dist.values,
}).set_index('Return Tier')
summary.style.background_gradient(subset=['Trade Count'], cmap='Blues')

,Description,Trade Count
Return Tier,,
1,Salary dump,39
2,Minimal,94
3,Depth,86
4,Mid-low,58
5,Mid,38
6,Mid-high,25
7,High,19
8,Very high,8
9,Elite package,1


In [3]:
pos_counts = complete.groupby('position_group').agg(
    Trades=('trade_id', 'count'),
    Avg_wWAR=('wWAR', 'mean'),
    Avg_Tier=('return_tier', 'mean'),
).round(2).sort_values('Trades', ascending=False)
pos_counts.columns = ['Trades', 'Avg wWAR', 'Avg Return Tier']
pos_counts

,Trades,Avg wWAR,Avg Return Tier
position_group,,,
RP,110,0.51,2.75
SP,86,2.05,4.34
OF,67,1.86,3.25
C,20,1.22,3.00
SS,19,2.46,3.47
1B,19,2.01,3.58
2B,18,2.19,3.56
3B,18,2.39,3.83
DH,4,2.00,3.50


## Case Studies

Three players illustrate how the model handles different profiles — from franchise-altering to solid depth. Each section shows the player's profile at the time of trade, what the model predicts, and what actually happened.

---

### Case Study 1: Juan Soto — The Franchise Player (Two Acts)

**August 2, 2022 — Washington → San Diego**

Juan Soto at 23 was already one of the best hitters in baseball. His 7.3 fWAR the prior season placed him in the top five players in MLB. Washington, in a rebuild, traded him at the deadline with three years of arbitration control remaining.

The question: what is a 23-year-old franchise hitter worth at the trade deadline?

In [4]:
DOLLARS_PER_WAR = 7.0
DISCOUNT_RATE   = 0.05
CONTROL_DISC    = 0.875

soto_22 = complete[complete['player_name'] == 'Juan Soto'].query('season == 2022').iloc[0]

def aging_delta(age):
    if age < 27:   return  0.25
    if age <= 30:  return  0.00
    if age <= 33:  return -0.50
    return -0.75

wwar   = float(soto_22['wWAR'])
age    = int(soto_22['age_at_trade'])
salary = float(soto_22['salary_m'])
years  = int(soto_22['years_control_remaining'])

rows = []
for i in range(years):
    yr  = 2022 + i
    war = max(0, wwar + (aging_delta(age + i) if i > 0 else 0))
    market  = war * DOLLARS_PER_WAR
    sal     = salary * (1.15 ** i)  # arb escalation estimate
    disc_surplus = (market - sal) / (1 + DISCOUNT_RATE) ** i
    rows.append({'Year': yr, 'Age': age + i, 'WAR': round(war, 1),
                 'Market Value': f'${market:.1f}M', 'Est. Salary': f'${sal:.1f}M',
                 'Disc. Surplus': f'${disc_surplus:.1f}M'})

total_disc = sum(float(r['Disc. Surplus'].replace('$', '').replace('M', '')) for r in rows)
trade_val  = total_disc * CONTROL_DISC
print(f"Profile: {int(soto_22['age_at_trade'])}yo | "
      f"{soto_22['contract_status'].upper()} | "
      f"wWAR {soto_22['wWAR']} | "
      f"${soto_22['salary_m']}M salary")
print(f"Total Discounted Surplus : ${total_disc:.1f}M")
print(f"Trade Value (x0.875)     : ${trade_val:.1f}M\n")
pd.DataFrame(rows).set_index('Year').style.set_caption(
    'Juan Soto — Surplus Value at Trade (2022)')

Profile: 23yo | ARB2 | wWAR 6.15 | $17.1M salary
Total Discounted Surplus : $70.0M
Trade Value (x0.875)     : $61.2M



,Age,WAR,Market Value,Est. Salary,Disc. Surplus
Year,,,,,
2022,23,6.200000,$43.1M,$17.1M,$26.0M
2023,24,6.400000,$44.8M,$19.7M,$23.9M
2024,25,6.400000,$44.8M,$22.6M,$20.1M


In [5]:
print(f"Return tier: {int(soto_22['return_tier'])}/10")
print(f"\nReturn summary: {soto_22['return_summary']}")
print(f"\nKey pieces: {soto_22['key_pieces']}")
print(f"\nNotes: {soto_22['notes']}")

Return tier: 8/10

Return summary: Franchise haul: 5 prospects incl. 2 future All-Stars

Key pieces: MacKenzie Gore,CJ Abrams,Robert Hassell III,James Wood,Jarlin Suarez

Notes: Josh Bell included (3.1 WAR 1B last year of control at 10 m); Soto pending arb comp case; return_tier adjusted from 10 to 8 to isolate Soto's Value


**December 1, 2023 — San Diego → New York**

Sixteen months later, the Padres moved Soto again — this time with just **one year of control** remaining before free agency. Same player. Very different negotiating position.

The rental market is one of the most consistent patterns in baseball: teams pay a steep premium for sustained control, and dramatically less for a one-year rental. The model quantifies exactly how steep that discount is.

In [6]:
soto_23 = complete[complete['player_name'].str.contains('Juan Soto')].query('season == 2024').iloc[0]

comparison = pd.DataFrame([
    {
        'Trade': 'WSN → SDP (Aug 2022)',
        'Age': int(soto_22['age_at_trade']),
        'wWAR': float(soto_22['wWAR']),
        'Contract': soto_22['contract_status'].upper(),
        'Yrs Control': int(soto_22['years_control_remaining']),
        'Salary': f"${float(soto_22['salary_m']):.1f}M",
        'Return Tier': f"{int(soto_22['return_tier'])}/10",
        'Top Pieces': 'MacKenzie Gore, CJ Abrams, James Wood',
    },
    {
        'Trade': 'SDP → NYY (Dec 2023)',
        'Age': int(soto_23['age_at_trade']),
        'wWAR': float(soto_23['wWAR']),
        'Contract': soto_23['contract_status'].upper(),
        'Yrs Control': int(soto_23['years_control_remaining']),
        'Salary': f"${float(soto_23['salary_m']):.1f}M",
        'Return Tier': f"{int(soto_23['return_tier'])}/10",
        'Top Pieces': 'Michael King, Drew Thorpe',
    },
]).set_index('Trade')

comparison.T.style.set_caption('Juan Soto: Same Player, Two Very Different Trades')

Trade,WSN → SDP (Aug 2022),SDP → NYY (Dec 2023)
Age,23,25
wWAR,6.150000,5.490000
Contract,ARB2,ARB3
Yrs Control,3,1
Salary,$17.1M,$23.0M
Return Tier,8/10,7/10
Top Pieces,"MacKenzie Gore, CJ Abrams, James Wood","Michael King, Drew Thorpe"


The model explains the tier drop (9 → 7) precisely: the rental discount. With one year of control, the acquiring team has no leverage — they are pricing a single postseason run, not a franchise cornerstone. The trade value window collapses from ~$61M to ~$16M.

**Takeaway:** Years of control is often the single most important variable in any trade. The model captures this automatically through the discounted surplus calculation.

---

### Case Study 2: Mason Miller — Why WAR Undersells Elite Closers

**August 1, 2025 — Oakland → San Diego**

Mason Miller's raw wWAR of **1.7** looks like a solid depth arm — the kind that fetches a fringe prospect, not the #3 overall prospect in baseball.

But wWAR alone misses something critical for elite relievers: **leverage**. An elite closer pitches exclusively in the highest-leverage moments of a game. Their impact per inning is dramatically higher than a starter or middle reliever.

The model applies a **1.8× leverage multiplier** for closers (approximating their average game Leverage Index), turning 1.7 raw WAR into ~3.1 effective WAR. Combined with 4 years of control at $765K/yr, the surplus value is enormous.

In [ ]:
miller = complete[complete['player_name'].str.contains('Mason Miller')].iloc[0]

LEVERAGE = 1.8  # closer gmLI proxy
eff_war = float(miller['wWAR']) * LEVERAGE

rows = []
age    = int(miller['age_at_trade'])
salary = float(miller['salary_m'])
ctrl   = int(miller['years_control_remaining'])
for i in range(ctrl):
    yr  = 2025 + i
    war = max(0, eff_war + (aging_delta(age + i) if i > 0 else 0))
    market = war * DOLLARS_PER_WAR
    if i < 2:
        sal   = salary
        stype = 'Pre-arb'
    else:
        pct   = {2: 0.40, 3: 0.60, 4: 0.80}.get(i, 0.80)
        sal   = market * pct
        stype = f'Arb {i - 1} (est.)'
    disc_surplus = (market - sal) / (1 + DISCOUNT_RATE) ** i
    rows.append({'Year': yr, 'Age': age + i, 'WAR (lev.)': round(war, 1),
                 'Market': f'${market:.1f}M', 'Salary': f'${sal:.2f}M',
                 'Type': stype, 'Disc. Surplus': f'${disc_surplus:.1f}M'})

total_disc = sum(float(r['Disc. Surplus'].replace('$', '').replace('M', '')) for r in rows)
trade_val  = total_disc * CONTROL_DISC
print(f"Raw wWAR: {miller['wWAR']} → Leverage-adjusted: {eff_war:.1f}")
print(f"Pre-arb salary: ${salary:.3f}M | Years control: {ctrl}")
print(f"Total Discounted Surplus : ${total_disc:.1f}M")
print(f"Trade Value (x0.875)     : ${trade_val:.1f}M\n")
pd.DataFrame(rows).set_index('Year').style.set_caption(
    'Mason Miller — Surplus Value (Closer Leverage Applied)')

In [8]:
print(f"Return tier: {int(miller['return_tier'])}/10")
print(f"\nKey pieces: {miller['key_pieces']}")
print(f"\n{miller['return_summary']}")
print(f"\nNotes: {miller['notes']}")

Return tier: 7/10

Key pieces: Leodalis De Vries, Braden Nett, Henry Baez, Eduarniel Nunez

OAK received 18-year-old SS Leodalis De Vries (MLB.com No. 3 overall prospect, Baseball America No. 5), plus three Padres arms: RHP Braden Nett (SD's #3 prospect), RHP Henry Baez (SD's #13 prospect), and RHP Eduarniel Nunez (SD's #17 prospect). De Vries was the clear headliner and one of the top prospects in baseball at the time.

Notes: Miller was co-traded alongside SP JP Sears in a single package deal on July 31, 2025. Miller was the primary asset  elite closer averaging 101+ mph with 20 saves and a 3.76 ERA in 38 appearances at time of trade. Pre-arb status gave SD 5 years of control at below-market cost. Miller became Super 2 arbitration eligible after the 2025 season and signed a 1-year/$4M deal with SD for 2026. Return tier adjusted down 1 from 8 because Sears was bundled in the same deal.


Leodalis De Vries was the #3 prospect in baseball at the time of the trade. Four pieces total. For a player with a raw wWAR of 1.7.

**Takeaway:** WAR metrics are calibrated on a per-inning basis for starters. An elite closer in a high-leverage role generates surplus value that the raw number undersells. The leverage adjustment is not optional for reliever valuation — it is the difference between pricing a depth arm and pricing a franchise-caliber trade chip.

---

### Case Study 3: Christian Yelich — When the Contract Is the Story

**January 25, 2018 — Miami → Milwaukee**

Christian Yelich at 26 was a legitimate star — four consecutive seasons of 3+ WAR, Gold Glove-caliber outfielder, coming off a 4.2 fWAR season. The Marlins were trading him in a payroll dump under new ownership (Derek Jeter), not because they doubted his talent.

The question the model asks: what does a 4-WAR player at **$7M/yr** actually cost to acquire?

Milwaukee gave up Lewis Brinson (Milwaukee's #1 prospect, #13 nationally per MLB Pipeline), Isan Diaz (#86 nationally), Monte Harrison, and Jordan Yamamoto. At the time, the reaction was **mixed — not uniformly negative.** Brinson was a real top-15 national prospect, and several analysts called the return fair. The criticism came later, after Yelich won the 2018 NL MVP and Brinson hit .183 in Miami.

The model explains both sides.

In [ ]:
yelich = complete[complete['player_name'].str.contains('Christian Yelich')].query('season == 2018').iloc[0]

# Actual salary schedule from the 2015 extension (7yr/$49.57M)
YELICH_SALARIES = {2018: 7.0, 2019: 7.5, 2020: 12.5, 2021: 14.0}

wwar  = float(yelich['wWAR'])
age   = int(yelich['age_at_trade'])
years = int(yelich['years_control_remaining'])

rows = []
for i in range(years):
    yr     = 2018 + i
    war    = max(0, wwar + (aging_delta(age + i) if i > 0 else 0))
    market = war * DOLLARS_PER_WAR
    sal    = YELICH_SALARIES.get(yr, 14.0)
    disc_surplus = (market - sal) / (1 + DISCOUNT_RATE) ** i
    rows.append({'Year': yr, 'Age': age + i, 'WAR': round(war, 1),
                 'Market Value': f'${market:.1f}M', 'Contracted Salary': f'${sal:.1f}M',
                 'Disc. Surplus': f'${disc_surplus:.1f}M'})

total_disc = sum(float(r['Disc. Surplus'].replace('$','').replace('M','')) for r in rows)
trade_val  = total_disc * CONTROL_DISC
market_rate_yr1 = wwar * DOLLARS_PER_WAR
print(f"Profile: {age}yo | SIGNED | wWAR {wwar} | ${YELICH_SALARIES[2018]}M salary (2018)")
print(f"Market value of {wwar} WAR    : ${market_rate_yr1:.1f}M/yr")
print(f"Contract subsidy (yr 1)  : ${market_rate_yr1 - YELICH_SALARIES[2018]:.1f}M below market")
print(f"Total Discounted Surplus : ${total_disc:.1f}M")
print(f"Trade Value (×0.875)     : ${trade_val:.1f}M\n")
pd.DataFrame(rows).set_index('Year').style.set_caption(
    'Christian Yelich — Surplus Value at Trade (2018)')

In [ ]:
print(f"Return tier: {int(yelich['return_tier'])}/10")
print(f"\nReturn summary: {yelich['return_summary']}")
print(f"\nKey pieces: {yelich['key_pieces']}")
print(f"\nNotes: {yelich['notes']}")

The model output and the market agreed: tier 8. A top-15 national prospect plus three supporting pieces for a 26-year-old star.

**This is the contract working.** At market rate — roughly $28M/yr for a 4-WAR player — Yelich never gets traded. Miami could not have extracted a top-15 national prospect for a player costing $28M/yr. At $7M/yr, the acquiring team captures ~$21M/yr in surplus, and the market prices that into the return package.

The reaction-at-the-time context matters: the analysts who called it a fair trade were not wrong given what they knew. Brinson was a legitimate top-15 national prospect. What they could not predict was Yelich winning back-to-back MVPs and Brinson hitting .183.

**Takeaway:** Surplus value does not live in WAR alone. A 4-WAR player at $7M is a completely different trade asset than a 4-WAR player at $28M. The contract subsidy is what made Yelich tradeable — and what forced Milwaukee to pay with their best prospect.

---

## What the Model Tells Us

In [ ]:
case_studies = pd.DataFrame([
    {
        'Player': 'Juan Soto', 'Trade': 'WSN → SDP', 'Date': 'Aug 2022',
        'Age': int(soto_22['age_at_trade']),
        'wWAR': float(soto_22['wWAR']),
        'Yrs Ctrl': int(soto_22['years_control_remaining']),
        'Contract': soto_22['contract_status'].upper(),
        'Salary': f"${float(soto_22['salary_m']):.1f}M",
        'Model Signal': 'Franchise — top 1%',
        'Actual Tier': f"{int(soto_22['return_tier'])}/10",
        'Headline Return': '5 prospects incl. 2 future All-Stars',
    },
    {
        'Player': 'Juan Soto', 'Trade': 'SDP → NYY', 'Date': 'Dec 2023',
        'Age': int(soto_23['age_at_trade']),
        'wWAR': float(soto_23['wWAR']),
        'Yrs Ctrl': int(soto_23['years_control_remaining']),
        'Contract': soto_23['contract_status'].upper(),
        'Salary': f"${float(soto_23['salary_m']):.1f}M",
        'Model Signal': 'Rental discount — 1yr only',
        'Actual Tier': f"{int(soto_23['return_tier'])}/10",
        'Headline Return': 'Michael King + Drew Thorpe',
    },
    {
        'Player': 'Mason Miller', 'Trade': 'OAK → SDP', 'Date': 'Aug 2025',
        'Age': int(miller['age_at_trade']),
        'wWAR': float(miller['wWAR']),
        'Yrs Ctrl': int(miller['years_control_remaining']),
        'Contract': miller['contract_status'].upper(),
        'Salary': f"${float(miller['salary_m']):.3f}M",
        'Model Signal': 'Elite surplus (leverage adj.)',
        'Actual Tier': f"{int(miller['return_tier'])}/10",
        'Headline Return': 'De Vries (#3 overall) + 3 arms',
    },
    {
        'Player': 'Christian Yelich', 'Trade': 'MIA → MIL', 'Date': 'Jan 2018',
        'Age': int(yelich['age_at_trade']),
        'wWAR': float(yelich['wWAR']),
        'Yrs Ctrl': int(yelich['years_control_remaining']),
        'Contract': yelich['contract_status'].upper(),
        'Salary': f"${float(yelich['salary_m']):.1f}M",
        'Model Signal': 'Contract subsidy drives value (+3 adj.)',
        'Actual Tier': f"{int(yelich['return_tier'])}/10",
        'Headline Return': 'Brinson (#13 natl) + 3 prospects',
    },
]).set_index('Player')

case_studies.style.set_caption('Case Study Summary').set_table_styles(
    [{'selector': 'th', 'props': [('font-weight', 'bold')]}])

### About This Tool

The model uses **Baseball Reference bWAR** (annualized from YTD pace) as the primary WAR source, falling back to local FanGraphs projection files (THE BAT X / ZiPS) when live data is unavailable. FanGraphs leaderboard APIs have been blocked since mid-2026; **FanGraphs and Spotrac** remain the source for contract structure when accessible, with Baseball Reference salary pages as a tertiary fallback. The comps engine searches a hand-built database of **370 verified trades (2015–2026)** — every return tier graded on prospect rankings at the time of trade, no hindsight adjustments.

Tier thresholds, development discounts, and WAR-floor caps were calibrated against all 370 trades (back-test r = 0.538, MAE = 1.60 tiers). Pre-arb and arb1 tiers carry the largest remaining uncertainty; comps are the more reliable signal for those profiles.

*Model built as part of a data science portfolio targeting front office analytics roles.*